# FTDI 232H + nRF2401+

In [1]:
# nRF24L01(+) TX with FT232H (pyftdi only) — Jupyter-friendly, Ctrl+C clean exit
#
# Wiring (FT232H ADBUS -> nRF24):
#   D0/ADBUS0 = SCK
#   D1/ADBUS1 = MOSI
#   D2/ADBUS2 = MISO
#   D3/ADBUS3 = CSN  (CS0)
#   D4/ADBUS4 = CE   (GPIO)
#   GND       = GND
#   3.3V      = VCC  (or 5V -> adapter board -> 3.3V to nRF24)
#
# Tip: add 10–100 uF + 0.1 uF near the nRF24 VCC/GND.

import sys
import time
from pyftdi.spi import SpiController

# -----------------------------
# User settings
# -----------------------------
FTDI_URL   = "ftdi://ftdi:232h/1"
SPI_FREQ   = 1_000_000      # start conservative; 4_000_000 often works too
PIN_CE     = 4              # ADBUS4 / D4
CS_INDEX   = 0              # CS0 = ADBUS3 / D3
TX_HZ      = 20             # transmit rate
CHANNEL    = 5            # 2400 + CHANNEL MHz => 2476 MHz
ADDR       = b"\xE7"*5      # 5-byte address
DATA_RATE  = "250k"         # "250k", "1M", "2M"
PA_LEVEL   = "MAX"          # "MIN","LOW","HIGH","MAX"
PAYLOAD_SZ = 32             # fixed payload size (1..32); we pad/truncate

# -----------------------------
# nRF24 command constants
# -----------------------------
R_REGISTER    = 0x00  # SPI command: read register
W_REGISTER    = 0x20  # SPI command: write register
REGISTER_MASK = 0x1F  # Mask for register address (lower 5 bits)

W_TX_PAYLOAD  = 0xA0  # SPI command: write TX payload to FIFO
FLUSH_TX      = 0xE1  # SPI command: flush TX FIFO
NOP           = 0xFF  # SPI no-op (returns STATUS)

# -----------------------------
# nRF24 registers
# -----------------------------
CONFIG      = 0x00  # Configuration (power, RX/TX mode, CRC)
EN_AA       = 0x01  # Enable auto-acknowledge per data pipe
EN_RXADDR   = 0x02  # Enable RX addresses (pipes)
SETUP_AW    = 0x03  # Address width setup (3–5 bytes)
SETUP_RETR  = 0x04  # Auto retransmit delay/count
RF_CH       = 0x05  # RF channel (2400 MHz + value)
RF_SETUP    = 0x06  # RF data rate and PA power
STATUS      = 0x07  # Status flags (IRQ sources, pipe number)
OBSERVE_TX  = 0x08  # TX retransmit counters
RX_ADDR_P0  = 0x0A  # RX address for pipe 0
TX_ADDR     = 0x10  # Transmit address
RX_PW_P0    = 0x11  # RX payload width for pipe 0
FIFO_STATUS = 0x17  # TX/RX FIFO status flags
DYNPD       = 0x1C  # Dynamic payload enable
FEATURE     = 0x1D  # Extra features (dyn payload, ACK payload)

# -----------------------------
# STATUS register bits
# -----------------------------
RX_DR  = 1 << 6  # Data received interrupt
TX_DS  = 1 << 5  # Data sent interrupt
MAX_RT = 1 << 4  # Max retransmits reached

# -----------------------------
# CONFIG register bits
# -----------------------------
PWR_UP  = 1 << 1  # Power up the radio
PRIM_RX = 1 << 0  # Primary RX mode (0 = TX, 1 = RX)
EN_CRC  = 1 << 3  # Enable CRC
CRCO    = 1 << 2  # CRC length (0 = 1 byte, 1 = 2 bytes)


def _rf_setup_value(data_rate: str, pa_level: str) -> int:
    # RF_SETUP bits (nRF24L01+):
    #  bit5 RF_DR_LOW (1 => 250kbps)
    #  bit3 RF_DR_HIGH (1 => 2Mbps)
    #  bits2:1 RF_PWR (00=-18, 01=-12, 10=-6, 11=0dBm)
    dr_low = 0
    dr_high = 0
    if data_rate.lower() in ("250k", "250kbps", "0.25m"):
        dr_low = 1
        dr_high = 0
    elif data_rate.lower() in ("1m", "1mbps"):
        dr_low = 0
        dr_high = 0
    elif data_rate.lower() in ("2m", "2mbps"):
        dr_low = 0
        dr_high = 1
    else:
        raise ValueError("DATA_RATE must be one of: '250k', '1M', '2M'")

    pwr_map = {"MIN": 0b00, "LOW": 0b01, "HIGH": 0b10, "MAX": 0b11}
    if pa_level.upper() not in pwr_map:
        raise ValueError("PA_LEVEL must be one of: 'MIN','LOW','HIGH','MAX'")
    pwr = pwr_map[pa_level.upper()]

    val = 0
    val |= (dr_low  << 5)
    val |= (dr_high << 3)
    val |= (pwr     << 1)
    return val

class NRF24Tx:
    def __init__(self, ftdi_url: str, spi_freq: int, pin_ce: int, cs_index: int):
        self.pin_ce = pin_ce

        self.spi = SpiController()
        self.spi.configure(ftdi_url)
        self.port = self.spi.get_port(cs=cs_index, freq=spi_freq, mode=0)

        # IMPORTANT: use the *same* SpiController for GPIO to avoid FTDI pin conflicts
        self.gpio = self.spi.get_gpio()
        self.gpio.set_direction(1 << self.pin_ce, 1 << self.pin_ce)

        self._gpio_state = 0x00
        self.ce(0)

    def close(self):
        try:
            self.ce(0)
        finally:
            self.spi.terminate()

    def ce(self, level: int):
        if level:
            self._gpio_state |= (1 << self.pin_ce)
        else:
            self._gpio_state &= ~(1 << self.pin_ce)
        self.gpio.write(self._gpio_state)

    def _xfer(self, data: bytes) -> bytes:
        return self.port.exchange(data, duplex=True)

    def read_reg(self, reg: int, n: int = 1) -> bytes:
        cmd = bytes([R_REGISTER | (reg & REGISTER_MASK)]) + bytes([NOP] * n)
        resp = self._xfer(cmd)
        return resp[1:]

    def write_reg(self, reg: int, value):
        if isinstance(value, int):
            value = bytes([value])
        cmd = bytes([W_REGISTER | (reg & REGISTER_MASK)]) + value
        self._xfer(cmd)

    def cmd(self, opcode: int):
        self._xfer(bytes([opcode]))

    def get_status(self) -> int:
        return self.read_reg(STATUS, 1)[0]

    def clear_irqs(self):
        # Write 1s to clear
        self.write_reg(STATUS, RX_DR | TX_DS | MAX_RT)

    def observe_tx(self) -> tuple[int, int, int]:
        # Returns (raw, PLOS_CNT, ARC_CNT)
        v = self.read_reg(OBSERVE_TX, 1)[0]
        return v, (v >> 4) & 0x0F, v & 0x0F

    def configure_tx(self,
                     channel: int,
                     addr: bytes,
                     payload_size: int,
                     data_rate: str,
                     pa_level: str,
                     crc_bytes: int = 2):
        if not (0 <= channel <= 125):
            raise ValueError("channel must be 0..125")
        if len(addr) != 5:
            raise ValueError("addr must be exactly 5 bytes")
        if not (1 <= payload_size <= 32):
            raise ValueError("payload_size must be 1..32")
        if crc_bytes not in (1, 2):
            raise ValueError("crc_bytes must be 1 or 2")

        self.ce(0)  # keep CE low during config

        # Basic: fixed payloads, no dynamic payload, no ack/retries
        self.write_reg(EN_AA, 0x00)       # disable auto-ack (simple TX)
        self.write_reg(SETUP_RETR, 0x00)  # disable retransmits
        self.write_reg(FEATURE, 0x00)     # disable extra features
        self.write_reg(DYNPD, 0x00)       # disable dynamic payloads

        # 5-byte addresses
        self.write_reg(SETUP_AW, 0x03)

        # RF settings
        self.write_reg(RF_CH, channel)
        self.write_reg(RF_SETUP, _rf_setup_value(data_rate, pa_level))

        # Addresses (pipe0 + TX)
        self.write_reg(TX_ADDR, addr)
        self.write_reg(RX_ADDR_P0, addr)
        self.write_reg(EN_RXADDR, 0x01)  # enable pipe0 (harmless in TX-only)

        # Fixed payload width for pipe0
        self.write_reg(RX_PW_P0, payload_size)

        # CONFIG: power up, TX mode (PRIM_RX=0), CRC enable
        cfg = PWR_UP | EN_CRC | (CRCO if crc_bytes == 2 else 0)
        self.write_reg(CONFIG, cfg)
        time.sleep(0.005)  # allow oscillator to start

        # Clear & flush
        self.clear_irqs()
        self.cmd(FLUSH_TX)

    def send(self, payload: bytes, ce_high_ms: float = 2.0) -> dict:
        """
        Send one payload. Returns a dict with status/fifo/observe.
        In no-ack mode, TX_DS should set when packet sent.
        """
        # pad/truncate to fixed size (nRF24 supports 1..32 fixed length)
        if len(payload) < PAYLOAD_SZ:
            payload = payload + b"\x00" * (PAYLOAD_SZ - len(payload))
        else:
            payload = payload[:PAYLOAD_SZ]

        # clear flags and flush TX so each send is clean & SDR-friendly
        self.clear_irqs()
        self.cmd(FLUSH_TX)

        # load payload
        self._xfer(bytes([W_TX_PAYLOAD]) + payload)

        # Start TX: hold CE high a bit (ms scale is fine from PC)
        self.ce(1)
        time.sleep(ce_high_ms / 1000.0)
        self.ce(0)

        # Read status & fifo after TX
        st = self.get_status()
        fs = self.read_reg(FIFO_STATUS, 1)[0]
        obs_raw, plos, arc = self.observe_tx()

        # Clear any latched flags for next loop
        self.clear_irqs()

        return {
            "STATUS": st,
            "TX_DS": bool(st & TX_DS),
            "MAX_RT": bool(st & MAX_RT),
            "FIFO_STATUS": fs,
            "OBSERVE_TX": obs_raw,
            "PLOS_CNT": plos,
            "ARC_CNT": arc,
        }

# -----------------------------
# Run TX loop (Ctrl+C to exit)
# -----------------------------
radio = NRF24Tx(FTDI_URL, SPI_FREQ, PIN_CE, CS_INDEX)

radio.configure_tx(
    channel=CHANNEL,
    addr=ADDR,
    payload_size=PAYLOAD_SZ,
    data_rate=DATA_RATE,
    pa_level=PA_LEVEL,
    crc_bytes=2,
)

period = 1.0 / TX_HZ
count = 0
t_next = time.monotonic()

print(f"nRF24 TX running @ {TX_HZ} Hz | CH={CHANNEL} ({2400+CHANNEL} MHz) | DR={DATA_RATE} | PA={PA_LEVEL}")
print("Press Ctrl+C to stop.\n")

try:
    while True:
        # Example payload (easy to spot if you later add a receiver)
        msg = f"pkt {count}".encode("ascii")
        msg = bytes.fromhex("0000000000e21eaa5efb0f7e5f789fb53f5c394c11326e")

        info = radio.send(msg, ce_high_ms=1.0)
        # print(f"{count:06d}  TX_DS={int(info['TX_DS'])}  MAX_RT={int(info['MAX_RT'])}  "
        #       f"STATUS=0x{info['STATUS']:02x}  FIFO=0x{info['FIFO_STATUS']:02x}  "
        #       f"OBS=0x{info['OBSERVE_TX']:02x}")

        sys.stdout.write(f"\r{count:06d}  TX_DS={int(info['TX_DS'])}  MAX_RT={int(info['MAX_RT'])}  "
              f"STATUS=0x{info['STATUS']:02x}  FIFO=0x{info['FIFO_STATUS']:02x}  "
              f"OBS=0x{info['OBSERVE_TX']:02x}")
        sys.stdout.flush()

        count += 1

        # keep a steady rate using monotonic scheduling
        t_next += period
        sleep_s = t_next - time.monotonic()
        if sleep_s > 0:
            time.sleep(sleep_s)
        else:
            # we're behind; resync
            t_next = time.monotonic()

except KeyboardInterrupt:
    print("\nCtrl+C received — shutting down cleanly...")

finally:
    radio.close()
    print("FT232H released, CE low, done.")


nRF24 TX running @ 20 Hz | CH=5 (2405 MHz) | DR=250k | PA=MAX
Press Ctrl+C to stop.

000023  TX_DS=1  MAX_RT=0  STATUS=0x2e  FIFO=0x11  OBS=0x03
Ctrl+C received — shutting down cleanly...
FT232H released, CE low, done.


# Channel Hopping Tx

In [1]:
# nRF24L01(+) TX with FT232H (pyftdi only) — Jupyter-friendly, Ctrl+C clean exit
#
# Wiring (FT232H ADBUS -> nRF24):
#   D0/ADBUS0 = SCK
#   D1/ADBUS1 = MOSI
#   D2/ADBUS2 = MISO
#   D3/ADBUS3 = CSN  (CS0)
#   D4/ADBUS4 = CE   (GPIO)
#   GND       = GND
#   3.3V      = VCC  (or 5V -> adapter board -> 3.3V to nRF24)
#
# Tip: add 10–100 uF + 0.1 uF near the nRF24 VCC/GND.

import sys
import time
from dataclasses import dataclass
from pyftdi.spi import SpiController

# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class TxConfig:
    ftdi_url: str = "ftdi://ftdi:232h/1"
    spi_freq: int = 4_000_000         # 1_000_000 if you need to be conservative
    cs_index: int = 0                 # CS0 = ADBUS3 / D3
    ce_pin: int = 4                   # ADBUS4 / D4

    channels: tuple[int, int] = (5, 50)   # hop between these two channels (0..125)
    hop_gap_s: float = 0.001              # gap between end (CE low) -> next start

    addr: bytes = b"\xE7" * 5             # 5-byte address
    payload_sz: int = 32                  # fixed payload (1..32)

    data_rate: str = "250k"               # "250k", "1M", "2M"
    pa_level: str = "LOW"                 # "MIN","LOW","HIGH","MAX"
    crc_bytes: int = 2                    # 1 or 2

    ce_high_ms: float = 0.1               # duration CE stays high per TX

# -----------------------------
# nRF24 SPI commands
# -----------------------------
R_REGISTER    = 0x00
W_REGISTER    = 0x20
REGISTER_MASK = 0x1F

W_TX_PAYLOAD  = 0xA0
FLUSH_TX      = 0xE1
NOP           = 0xFF

# -----------------------------
# nRF24 registers
# -----------------------------
CONFIG      = 0x00
EN_AA       = 0x01
EN_RXADDR   = 0x02
SETUP_AW    = 0x03
SETUP_RETR  = 0x04
RF_CH       = 0x05
RF_SETUP    = 0x06
STATUS      = 0x07
OBSERVE_TX  = 0x08
RX_ADDR_P0  = 0x0A
TX_ADDR     = 0x10
RX_PW_P0    = 0x11
FIFO_STATUS = 0x17
DYNPD       = 0x1C
FEATURE     = 0x1D

# STATUS bits
RX_DR  = 1 << 6
TX_DS  = 1 << 5
MAX_RT = 1 << 4

# CONFIG bits
PWR_UP  = 1 << 1
PRIM_RX = 1 << 0
EN_CRC  = 1 << 3
CRCO    = 1 << 2


def rf_setup_value(data_rate: str, pa_level: str) -> int:
    # RF_SETUP bits (nRF24L01+):
    #  bit5 RF_DR_LOW  (1 => 250kbps)
    #  bit3 RF_DR_HIGH (1 => 2Mbps)
    #  bits2:1 RF_PWR  (00=-18, 01=-12, 10=-6, 11=0dBm)
    dr = data_rate.lower()
    if dr in ("250k", "250kbps", "0.25m"):
        dr_low, dr_high = 1, 0
    elif dr in ("1m", "1mbps"):
        dr_low, dr_high = 0, 0
    elif dr in ("2m", "2mbps"):
        dr_low, dr_high = 0, 1
    else:
        raise ValueError("data_rate must be one of: '250k', '1M', '2M'")

    pwr_map = {"MIN": 0b00, "LOW": 0b01, "HIGH": 0b10, "MAX": 0b11}
    try:
        pwr = pwr_map[pa_level.upper()]
    except KeyError as e:
        raise ValueError("pa_level must be one of: 'MIN','LOW','HIGH','MAX'") from e

    return (dr_low << 5) | (dr_high << 3) | (pwr << 1)


class NRF24Tx:
    def __init__(self, cfg: TxConfig):
        self.cfg = cfg
        self._gpio_state = 0x00
        self._channel: int | None = None

        self.spi = SpiController()
        self.spi.configure(cfg.ftdi_url)
        self.port = self.spi.get_port(cs=cfg.cs_index, freq=cfg.spi_freq, mode=0)

        self.gpio = self.spi.get_gpio()
        self.gpio.set_direction(1 << cfg.ce_pin, 1 << cfg.ce_pin)

        self.ce(0)

    # ---- GPIO ----
    def ce(self, level: int):
        if level:
            self._gpio_state |= (1 << self.cfg.ce_pin)
        else:
            self._gpio_state &= ~(1 << self.cfg.ce_pin)
        self.gpio.write(self._gpio_state)

    # ---- SPI helpers ----
    def _xfer(self, data: bytes) -> bytes:
        return self.port.exchange(data, duplex=True)

    def read_reg(self, reg: int, n: int = 1) -> bytes:
        cmd = bytes([R_REGISTER | (reg & REGISTER_MASK)]) + bytes([NOP] * n)
        resp = self._xfer(cmd)
        return resp[1:]

    def write_reg(self, reg: int, value: int | bytes):
        if isinstance(value, int):
            value = bytes([value])
        self._xfer(bytes([W_REGISTER | (reg & REGISTER_MASK)]) + value)

    def cmd(self, opcode: int):
        self._xfer(bytes([opcode]))

    # ---- Status ----
    def get_status(self) -> int:
        return self.read_reg(STATUS, 1)[0]

    def clear_irqs(self):
        self.write_reg(STATUS, RX_DR | TX_DS | MAX_RT)

    def observe_tx(self) -> int:
        return self.read_reg(OBSERVE_TX, 1)[0]

    # ---- Radio config ----
    def set_channel(self, channel: int):
        if self._channel == channel:
            return
        self.ce(0)
        self.write_reg(RF_CH, channel)
        self._channel = channel

    def configure_tx(self):
        c = self.cfg
        self.ce(0)

        # Disable features we don't use (simple unacked TX)
        self.write_reg(EN_AA, 0x00)
        self.write_reg(SETUP_RETR, 0x00)
        self.write_reg(FEATURE, 0x00)
        self.write_reg(DYNPD, 0x00)

        # 5-byte address
        self.write_reg(SETUP_AW, 0x03)

        self.set_channel(c.channels[0])
        self.write_reg(RF_SETUP, rf_setup_value(c.data_rate, c.pa_level))

        # Set TX address + pipe0 (used for ACK path, even if AA disabled)
        self.write_reg(TX_ADDR, c.addr)
        self.write_reg(RX_ADDR_P0, c.addr)
        self.write_reg(EN_RXADDR, 0x01)
        self.write_reg(RX_PW_P0, c.payload_sz)

        cfg = PWR_UP | EN_CRC | (CRCO if c.crc_bytes == 2 else 0)
        self.write_reg(CONFIG, cfg)
        time.sleep(0.005)  # power-up settling time

        self.clear_irqs()
        self.cmd(FLUSH_TX)

    # ---- Send ----
    def send(self, payload: bytes, *, channel: int | None = None) -> dict:
        c = self.cfg
        if len(payload) < c.payload_sz:
            payload += b"\x00" * (c.payload_sz - len(payload))
        else:
            payload = payload[: c.payload_sz]

        self.ce(0)
        if channel is not None:
            self.set_channel(channel)

        self.clear_irqs()
        self.cmd(FLUSH_TX)

        self._xfer(bytes([W_TX_PAYLOAD]) + payload)

        self.ce(1)
        time.sleep(c.ce_high_ms / 1000.0)
        self.ce(0)

        st = self.get_status()
        fs = self.read_reg(FIFO_STATUS, 1)[0]
        obs = self.observe_tx()

        self.clear_irqs()

        return {
            "STATUS": st,
            "TX_DS": bool(st & TX_DS),
            "MAX_RT": bool(st & MAX_RT),
            "FIFO_STATUS": fs,
            "OBSERVE_TX": obs,
            "CHANNEL": self._channel,
        }

    def close(self):
        self.ce(0)
        self.spi.terminate()

In [2]:
# -----------------------------
# Configuration
# -----------------------------
@dataclass(frozen=True)
class TxConfig:
    ftdi_url: str = "ftdi://ftdi:232h/1"
    spi_freq: int = 4_000_000         # 1_000_000 if you need to be conservative
    cs_index: int = 0                 # CS0 = ADBUS3 / D3
    ce_pin: int = 4                   # ADBUS4 / D4

    channels: tuple[int, int] = (5, 50)   # hop between these two channels (0..125)
    hop_gap_s: float = 0.001              # gap between end (CE low) -> next start

    addr: bytes = b"\xE7" * 5             # 5-byte address
    payload_sz: int = 32                  # fixed payload (1..32)

    data_rate: str = "250k"               # "250k", "1M", "2M"
    pa_level: str = "LOW"                 # "MIN","LOW","HIGH","MAX"
    crc_bytes: int = 2                    # 1 or 2

    ce_high_ms: float = 0.1               # duration CE stays high per TX

def main():
    cfg = TxConfig()

    payload = bytes.fromhex("0000000000e21eaa5efb0f7e5f789fb53f5c394c11326e")

    radio = NRF24Tx(cfg)
    radio.configure_tx()

    count = 0
    chan_i = 0

    print(
        f"nRF24 TX hopping {cfg.channels} | gap={cfg.hop_gap_s*1000:.3f} ms "
        f"(CE low -> next CE high)"
    )
    print("Press Ctrl+C to stop.\n")

    try:
        t_next_start = time.monotonic()
        while True:
            ch = cfg.channels[chan_i & 1]
            chan_i += 1

            now = time.monotonic()
            sleep_s = t_next_start - now
            if sleep_s > 0:
                time.sleep(sleep_s)
            else:
                t_next_start = now  # resync if behind

            info = radio.send(payload, channel=ch)

            # enforce hop gap after CE goes low (inside send())
            t_next_start = time.monotonic() + cfg.hop_gap_s

            sys.stdout.write(
                f"\r{count:06d}  CH={info['CHANNEL']:3d}  TX_DS={int(info['TX_DS'])}  "
                f"MAX_RT={int(info['MAX_RT'])}  STATUS=0x{info['STATUS']:02x}  "
                f"FIFO=0x{info['FIFO_STATUS']:02x}  OBS=0x{info['OBSERVE_TX']:02x}"
            )
            sys.stdout.flush()
            count += 1

    except KeyboardInterrupt:
        print("\nCtrl+C received — shutting down cleanly...")

    finally:
        radio.close()
        print("FT232H released, CE low, done.")


if __name__ == "__main__":
    main()

nRF24 TX hopping (5, 50) | gap=1.000 ms (CE low -> next CE high)
Press Ctrl+C to stop.

000670  CH=  5  TX_DS=0  MAX_RT=0  STATUS=0x0e  FIFO=0x01  OBS=0x03
Ctrl+C received — shutting down cleanly...
FT232H released, CE low, done.
